# 03: PatchCore ile YOLO karşılaştırması (metal_nut)

Soru: YOLO'nun etiketli kusurlu örneklerle PatchCore'a (0 etiket) yetişmesi için kaç etiket gerekir?
YOLO11n, MVTec maskelerinden üretilen kutularla 5, 10 ve 20 kusurlu örnekle eğitildi.
Değerlendirme: aynı 90 görüntü (68 kusurlu + 22 sağlam), image AUROC.

Veri seti: MVTec AD (CC BY-NC-SA 4.0). Bergmann et al., CVPR 2019.

In [ ]:
import torch
print(torch.cuda.is_available())

## 1. Kurulum
GPU kontrolü ve ultralytics kurulumu. COCO ile eğitilmiş yolo11n.pt ile test.

In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics
print(ultralytics.__version__)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # n = nano, en küçük ve hızlı model
sonuc = model("https://ultralytics.com/images/bus.jpg")
print(sonuc[0].boxes.cls, sonuc[0].boxes.conf)

In [ ]:
from pathlib import Path
import cv2
ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")

In [ ]:
def maskeden_kutular(mask_yolu):
    m = cv2.imread(str(mask_yolu), cv2.IMREAD_GRAYSCALE)
    h, w = m.shape
    _, m = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    konturlar, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    kutular = []
    for k in konturlar:
        x, y, bw, bh = cv2.boundingRect(k)
        if bw * bh < 20:          # çok küçük gürültüyü atla
            continue
        kutular.append(((x + bw/2)/w, (y + bh/2)/h, bw/w, bh/h))
    return kutular

mask = sorted((ROOT/"metal_nut/ground_truth/bent").glob("*.png"))[0]
print(mask.name)
print(maskeden_kutular(mask))

In [ ]:
import matplotlib.pyplot as plt

gor = ROOT/"metal_nut/test/bent"/(mask.name.replace("_mask", ""))
img = cv2.cvtColor(cv2.imread(str(gor)), cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]

for (xc, yc, bw, bh) in maskeden_kutular(mask):
    x1, y1 = int((xc - bw/2)*W), int((yc - bh/2)*H)
    x2, y2 = int((xc + bw/2)*W), int((yc + bh/2)*H)
    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 4)

plt.imshow(img); plt.axis("off"); plt.show()

In [ ]:
import random, shutil, json
from pathlib import Path
import cv2

ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
CAT = "metal_nut"
CIKTI = Path("/kaggle/working/yolo_veri")
random.seed(0)

def kutular(mask_yolu):          # Gün 16'daki fonksiyon
    m = cv2.imread(str(mask_yolu), cv2.IMREAD_GRAYSCALE)
    h, w = m.shape
    _, m = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    kon, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for k in kon:
        x, y, bw, bh = cv2.boundingRect(k)
        if bw * bh >= 20:
            out.append(((x+bw/2)/w, (y+bh/2)/h, bw/w, bh/h))
    return out

# Tüm kusurlu test görüntülerini topla (görüntü yolu, maske yolu)
kusurlu = []
for tur in sorted((ROOT/CAT/"test").iterdir()):
    if tur.name == "good":
        continue
    for g in sorted(tur.glob("*.png")):
        m = ROOT/CAT/"ground_truth"/tur.name/(g.stem + "_mask.png")
        kusurlu.append((g, m, tur.name))
random.shuffle(kusurlu)
print("toplam kusurlu:", len(kusurlu))

## 3. Eğitim setleri (n05, n10, n20)
Kusurlu görüntüler test klasöründen geldiği için eğitim ve doğrulamaya alınanlar değerlendirmeden çıkarıldı.
Her sete n kusurlu + n sağlam görüntü eklendi (sağlamlar boş etiketli).
Not: İlk denemede aynı adlı dosyalar birbirinin üstüne yazıldı (n10 → 8, n20 → 15 örnek).
Dosya adına kusur türü eklenerek düzeltildi ve eğitimler yeniden yapıldı.

In [ ]:
sagliam_tum = sorted((ROOT/CAT/"train/good").glob("*.png"))
random.shuffle(sagliam_tum)

def kur(n, val_sayisi=5):
    ad = f"n{n:02d}"
    kok = CIKTI/ad
    for s in ["train", "val"]:
        (kok/"images"/s).mkdir(parents=True, exist_ok=True)
        (kok/"labels"/s).mkdir(parents=True, exist_ok=True)

    egitim = kusurlu[:n]                      # n kusurlu eğitim
    val    = kusurlu[-val_sayisi:]            # son 5 kusurlu doğrulama (eğitimle karışmaz)
    sag_e  = sagliam_tum[:n]                  # n sağlam eğitim
    sag_v  = sagliam_tum[-val_sayisi:]        # 5 sağlam doğrulama

    def yaz(g, m, bolum, onek):
        hedef = kok/"images"/bolum/f"{onek}_{g.parent.name}_{g.name}"
        shutil.copy(g, hedef)
        etiket = kok/"labels"/bolum/(hedef.stem + ".txt")
        if m is None:
            etiket.write_text("")            # sağlam: boş etiket
        else:
            satirlar = [f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for x, y, w, h in kutular(m)]
            etiket.write_text("\n".join(satirlar))

    for g, m, _ in egitim: yaz(g, m, "train", "kus")
    for g, m, _ in val:    yaz(g, m, "val",   "kus")
    for g in sag_e:        yaz(g, None, "train", "sag")
    for g in sag_v:        yaz(g, None, "val",   "sag")

    (kok/"data.yaml").write_text(
        f"path: {kok}\ntrain: images/train\nval: images/val\nnames:\n  0: defect\n")

    json.dump([str(g) for g, _, _ in egitim],
              open(kok/"egitimde_kullanilan.json", "w"))
    return kok

for n in [5, 10, 20]:
    print(kur(n))

In [ ]:
!ls /kaggle/working/yolo_veri/n05/images/train | grep -c kus_
!ls /kaggle/working/yolo_veri/n10/images/train | grep -c kus_
!ls /kaggle/working/yolo_veri/n20/images/train | grep -c kus_
!ls /kaggle/working/yolo_veri/n20/images/val | grep -c kus_

## 4. YOLO eğitimi
yolo11n.pt üzerinde ince ayar: 100 epoch, patience=30, imgsz=640, batch=8.
Eğitim betik olarak çalıştırıldı (notebook içi ilerleme çubuğu hatasından kaçınmak için).

In [ ]:
!python yolo_egit.py 05 2>&1 | grep -E "BITTI"
!python yolo_egit.py 10 2>&1 | grep -E "BITTI"
!python yolo_egit.py 20 2>&1 | grep -E "BITTI"

In [ ]:
!ls /kaggle/working/yolo_veri/n05/images/train | grep -c kus_
!ls /kaggle/working/yolo_veri/n10/images/train | grep -c kus_
!ls /kaggle/working/yolo_veri/n20/images/train | grep -c kus_
!ls /kaggle/working/yolo_veri/n20/images/val | grep -c kus_

## 5. YOLO değerlendirmesi
Görüntü skoru = görüntüdeki en yüksek kutu güveni (kutu yoksa 0), conf=0.001.
| Etiketli örnek | Image AUROC |
|---|---|
| 5 | 0.428 |
| 10 | 0.359 |
| 20 | 0.408 |

In [ ]:
%%writefile yolo_degerlendir.py
import sys, json, random
from pathlib import Path
from sklearn.metrics import roc_auc_score
from ultralytics import YOLO

ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
CAT = "metal_nut"

random.seed(0)
kusurlu = []
for tur in sorted((ROOT/CAT/"test").iterdir()):
    if tur.name == "good":
        continue
    for g in sorted(tur.glob("*.png")):
        kusurlu.append(g)
random.shuffle(kusurlu)

kayit = json.load(open("/kaggle/working/yolo_veri/n20/egitimde_kullanilan.json"))
assert [str(g) for g in kusurlu[:20]] == kayit, "Sıra tutmuyor!"
print("sıra doğrulandı", flush=True)

hariç = set(kusurlu[:20]) | set(kusurlu[-5:])
kus_test = [g for g in kusurlu if g not in hariç]
sag_test = sorted((ROOT/CAT/"test/good").glob("*.png"))
print("hariç:", len(hariç), "kusurlu test:", len(kus_test), "sağlam test:", len(sag_test), flush=True)

dosyalar = kus_test + sag_test
etiket = [1]*len(kus_test) + [0]*len(sag_test)

for n in sys.argv[1:]:
    model = YOLO(f"/kaggle/working/yolo_sonuc/n{n}/weights/best.pt")
    skorlar = []
    for g in dosyalar:
        r = model(str(g), conf=0.001, verbose=False)[0]
        skorlar.append(float(r.boxes.conf.max()) if len(r.boxes) else 0.0)
    auc = roc_auc_score(etiket, skorlar)
    print("SONUC", f"n{n}", "image AUROC:", round(auc, 4), flush=True)
    json.dump({"n": n, "auroc": auc}, open(f"/kaggle/working/yolo_auc_n{n}.json", "w"))

In [ ]:
!python yolo_degerlendir.py 05 10 20 2>&1 | grep -E "sıra|hariç|SONUC|Error|Traceback"

## 6. Neden şans düzeyinde? (teşhis)
Ortalama skor: kusurlu 0.897, sağlam 0.915. Model her görüntüde (sağlam dahil) çok sayıda büyük, yüksek güvenli kutu çiziyor.
Kusuru değil, somunun kendisini "defect" olarak öğrenmiş görünüyor (yorum, kesin kanıt değil).

In [ ]:
%%writefile yolo_teshis.py
import json, random
from pathlib import Path
import numpy as np
from ultralytics import YOLO

ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
random.seed(0)
kusurlu = []
for tur in sorted((ROOT/"metal_nut"/"test").iterdir()):
    if tur.name == "good": continue
    for g in sorted(tur.glob("*.png")): kusurlu.append(g)
random.shuffle(kusurlu)
hariç = set(kusurlu[:20]) | set(kusurlu[-5:])
kus = [g for g in kusurlu if g not in hariç]
sag = sorted((ROOT/"metal_nut/test/good").glob("*.png"))

model = YOLO("/kaggle/working/yolo_sonuc/n20/weights/best.pt")
def skor(g):
    r = model(str(g), conf=0.001, verbose=False)[0]
    return float(r.boxes.conf.max()) if len(r.boxes) else 0.0

sk = np.array([skor(g) for g in kus]); ss = np.array([skor(g) for g in sag])
print("TESHIS kusurlu: ortalama", sk.mean().round(4), "sıfır skor sayısı", int((sk==0).sum()), "/", len(sk))
print("TESHIS saglam : ortalama", ss.mean().round(4), "sıfır skor sayısı", int((ss==0).sum()), "/", len(ss))

In [ ]:
!python yolo_teshis.py 2>&1 | grep TESHIS

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt, cv2
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
model = YOLO("/kaggle/working/yolo_sonuc/n20/weights/best.pt")

yollar = [sorted((ROOT/"metal_nut/test/good").glob("*.png"))[0],
          sorted((ROOT/"metal_nut/test/scratch").glob("*.png"))[0]]
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for a, y in zip(ax, yollar):
    r = model(str(y), conf=0.25, verbose=False)[0]
    a.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); a.axis("off")
    a.set_title(f"{y.parent.name}: {len(r.boxes)} kutu")
plt.show()

## 7. PatchCore, aynı alt kümede
Adil karşılaştırma için PatchCore aynı 90 görüntüde (68 kusurlu + 22 sağlam) ölçüldü: image AUROC ≈ 1.0 (0 etiketle).

In [ ]:
%%writefile patchcore_altkume.py
import json, random
from pathlib import Path
import torch
from sklearn.metrics import roc_auc_score
from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine

ROOT = "/kaggle/input/datasets/ipythonx/mvtec-ad"
CAT = "metal_nut"

# YOLO değerlendirmesiyle aynı alt küme
random.seed(0)
kusurlu = []
for tur in sorted((Path(ROOT)/CAT/"test").iterdir()):
    if tur.name == "good":
        continue
    for g in sorted(tur.glob("*.png")):
        kusurlu.append(g)
random.shuffle(kusurlu)
hariç = set(str(g) for g in kusurlu[:20]) | set(str(g) for g in kusurlu[-5:])

dm = MVTecAD(root=ROOT, category=CAT, train_batch_size=16, eval_batch_size=16, num_workers=2)
model = Patchcore()
engine = Engine(default_root_dir="/kaggle/working/results/altkume")
engine.fit(datamodule=dm, model=model)
tahm = engine.predict(model=model, dataloaders=dm.test_dataloader())

skor, etiket, yol = [], [], []
for b in tahm:
    skor   += [float(s) for s in b.pred_score.flatten().cpu()]
    etiket += [int(l) for l in b.gt_label.flatten().cpu()]
    yol    += [str(p) for p in b.image_path]

s2, e2 = [], []
for s, e, p in zip(skor, etiket, yol):
    if p in hariç:
        continue
    s2.append(s); e2.append(e)

print("SONUC alt kume:", len(s2), "kusurlu:", sum(e2), "saglam:", len(e2)-sum(e2), flush=True)
print("SONUC PatchCore alt kume AUROC:", round(roc_auc_score(e2, s2), 4), flush=True)
json.dump({"auroc": roc_auc_score(e2, s2)}, open("/kaggle/working/patchcore_altkume.json", "w"))

In [ ]:
!pip install anomalib

In [ ]:
!python patchcore_altkume.py 2>&1 | grep -E "SONUC|Error|Traceback"

## 8. Ana grafik
Etiketli kusur sayısına karşı image AUROC.

**Sınırlar:** tek kategori (metal_nut), tek seed, 10 görüntülük doğrulama seti, ayarlanmamış eğitim parametreleri.
"Bu ayarlarda YOLO 20 etikete kadar yetişemedi" doğru, "YOLO her zaman zayıf" iddiası değil.

In [ ]:
import matplotlib.pyplot as plt

x = [0, 5, 10, 20]
yolo = [0.428, 0.359, 0.408]

plt.figure(figsize=(7, 4.5))
plt.axhline(1.0, color="tab:green", label="PatchCore (0 etiket): 1.00")
plt.plot([5, 10, 20], yolo, "o-", color="tab:red", label="YOLO11n (etiketli)")
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.6, label="Şans (0.5)")
plt.xticks(x)
plt.ylim(0.2, 1.05)
plt.xlabel("Etiketli kusurlu görüntü sayısı")
plt.ylabel("Image AUROC (metal_nut, 68 kusurlu + 22 sağlam)")
plt.title("PatchCore vs YOLO: etiket sayısına karşı AUROC")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/ana_grafik.png", dpi=150, bbox_inches="tight")
plt.show()